# 06 - Training Loop

这一节把整条线闭合起来：

```text
前向预测 -> 计算 loss -> zero_grad -> backward -> 按 grad 更新参数 -> loss 下降
```

这一节只抓一句话：

> `grad` 告诉参数往哪里会让 loss 变大，所以训练时沿着 `-grad` 方向走一点点。

这个“一点点”就是学习率 `learning_rate`。

<!-- xiao-preview:start -->
## 课前预习作业：先做一次参数更新

训练循环里最重要的一行就是参数更新。先用一个数字算一次，再看完整训练。


In [1]:
# TODO: 把 None 改成你算出来的结果。
# p.data = 2.0, p.grad = 3.0, learning_rate = 0.1
# 更新规则：p.data += -learning_rate * p.grad
preview_updated_value = None
preview_need_zero_grad = None      # 每轮 backward 前是否要清空旧梯度？填 True 或 False
preview_positive_grad_direction = None  # grad 为正时，参数应该往哪个方向走？填 -1 或 1


def qa_check_06_preview(updated_value, need_zero_grad, positive_grad_direction):
    values = [updated_value, need_zero_grad, positive_grad_direction]
    if any(v is None for v in values):
        print('请先填写 TODO：先算一次参数更新。')
        return
    assert abs(updated_value - 1.7) < 1e-9, f'更新后应是 1.7，但你填的是 {updated_value}'
    assert need_zero_grad is True, '每轮 backward 前需要 zero_grad，否则梯度会累加旧值。'
    assert positive_grad_direction == -1, 'grad 为正时，要沿 -grad 方向更新，所以方向记作 -1。'
    print('OK: 参数更新预习通过，可以进入训练循环。')


qa_check_06_preview(preview_updated_value, preview_need_zero_grad, preview_positive_grad_direction)


请先填写 TODO：先算一次参数更新。


<!-- xiao-preview:hint -->
<details>
<summary><strong>Show / Hide 提示</strong></summary>

`2.0 + -0.1 * 3.0 = 1.7`。  
训练想让 loss 下降，所以一般沿着 `-grad` 的方向走。

</details>


<!-- xiao-preview:answer -->
<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_updated_value = 1.7
preview_need_zero_grad = True
preview_positive_grad_direction = -1
```

</details>


## 训练步骤地图

| 步骤 | 代码 | 你要盯住什么 |
|---|---|---|
| forward | `loss()` | 当前预测和 loss |
| 清空旧梯度 | `model.zero_grad()` | 所有 `p.grad` 回到 0 |
| backward | `total_loss.backward()` | 每个参数得到新 `grad` |
| update | `p.data += -lr * p.grad` | 参数朝降低 loss 的方向动一点 |
| repeat | `for step in range(...)` | loss 整体下降 |

这张表就是训练循环的骨架。

## 0. Setup

In [2]:
from pathlib import Path
import sys
import random

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))

from micrograd.engine import Value
from micrograd.nn import MLP

random.seed(1337)
print('project root:', PROJECT_ROOT)


def qa_check_update_parameters(func):
    p = Value(10.0)
    p.grad = 3.0
    func([p], learning_rate=0.1)

    if abs(p.data - 10.0) < 1e-12:
        print('TODO: 参数还没有变化，请先实现 update_parameters。')
        return

    expected = 9.7
    assert abs(p.data - expected) < 1e-9, f'expected {expected}, got {p.data}'
    print('OK: update_parameters passed.')


project root: /Users/barry/IdeaProjects/llm/micrograd


## 1. 一个很小的数据集

输入是两个数字，目标输出是 `1` 或 `-1`。我们让 MLP 学会把输入映射到目标。

In [3]:
xs = [
    [2.0, 3.0],
    [3.0, -1.0],
    [0.5, 1.0],
    [1.0, 1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

for x, y in zip(xs, ys):
    print(x, '->', y)

[2.0, 3.0] -> 1.0
[3.0, -1.0] -> -1.0
[0.5, 1.0] -> -1.0
[1.0, 1.0] -> 1.0


## 2. 建一个小 MLP

`MLP(2, [4, 1])` 的意思：

```text
输入维度 2
隐藏层 4 个神经元
输出层 1 个神经元
```

In [4]:
random.seed(1337)
model = MLP(2, [4, 1])
print(model)
print('number of parameters:', len(model.parameters()))

MLP of [Layer of [ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2)], Layer of [LinearNeuron(4)]]
number of parameters: 17


## 3. 定义 loss：先把“错多少”变成一个数

训练前，模型只是随机猜。我们需要一个数字告诉它：**现在错得有多严重**。这个数字就是 `loss`。

这一节用最简单的平方误差。先不要看公式，先看一条样本：

```text
标准答案 ygt = 1.0
模型预测 yout = 0.4
误差 diff = yout - ygt = 0.4 - 1.0 = -0.6
平方误差 squared = diff ** 2 = (-0.6) ** 2 = 0.36
```

为什么要平方？

```text
预测偏大：diff 是正数
预测偏小：diff 是负数
平方以后都变成正数
错得越远，平方误差越大
```

多条样本时，就对每条都算一次平方误差，然后取平均：

$$
loss = \frac{1}{N}\sum_i (\hat{y}_i - y_i)^2
$$

把符号翻译成人话：

| 符号/变量 | 含义 |
|---|---|
| `ygt` | ground truth，标准答案 |
| `yout` | 模型输出的预测值 |
| `yout - ygt` | 预测和答案差多少 |
| `(yout - ygt) ** 2` | 这一条样本的平方误差 |
| `sum(losses) / len(losses)` | 所有样本的平均错误，也就是总 loss |




In [ ]:
# 先用普通 Python 数字算一条样本，不涉及 Value 和 backward。
toy_ygt = 1.0      # 标准答案
toy_yout = 0.4     # 模型预测

toy_diff = toy_yout - toy_ygt
toy_squared = toy_diff ** 2

print('ygt      =', toy_ygt)
print('yout     =', toy_yout)
print('diff     = yout - ygt =', toy_diff)
print('squared  = diff ** 2  =', toy_squared)



### 把一条样本扩展到 4 条样本

当前数据集里有 4 条训练样本：

```python
xs = [
    [2.0, 3.0],
    [3.0, -1.0],
    [0.5, 1.0],
    [1.0, 1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]
```

`loss()` 做的事情就是：

```text
1. 对每条 x 调一次 predict(x)，得到 ypred
2. 把 ypred 和 ys 一一配对
3. 每一对都算 (预测值 - 标准答案) ** 2
4. 把 4 个平方误差求平均
```

注意：这里还没有更新参数，只是在问当前模型“现在错多少”。



In [ ]:
def predict(xrow):
    # xrow 是普通 Python 数字列表，比如 [2.0, 3.0]
    # model 需要吃 Value，所以先把每个数字包成 Value。
    values = [Value(x) for x in xrow]
    return model(values)


def loss():
    # 1. 对每条输入都做一次预测。
    ypred = [predict(x) for x in xs]

    # 2. 把标准答案 ys 和预测值 ypred 一一配对。
    # 3. 每条样本都算一个平方误差。
    losses = [(yout - ygt) ** 2 for ygt, yout in zip(ys, ypred)]

    # 4. 求平均：sum(losses) / 样本数量。
    total = sum(losses) * (1.0 / len(losses))
    return total, ypred


def loss_with_breakdown():
    ypred = [predict(x) for x in xs]
    losses = []

    print('i | x | ygt | yout | diff = yout-ygt | squared')
    print('--|---|-----|------|------------------|--------')

    for i, (xrow, ygt, yout) in enumerate(zip(xs, ys, ypred)):
        diff = yout - ygt
        squared = diff ** 2
        losses.append(squared)
        print(
            f'{i} | {xrow} | {ygt:+.1f} | {yout.data:+.4f} | '
            f'{diff.data:+.4f} | {squared.data:.4f}'
        )

    total = sum(losses) * (1.0 / len(losses))
    print('sum squared errors =', round(sum(item.data for item in losses), 4))
    print('average loss       =', total.data)
    return total, ypred, losses


initial_loss, initial_preds, initial_sample_losses = loss_with_breakdown()
print('initial predictions:', [round(p.data, 4) for p in initial_preds])



### 对照代码里的 4 行核心逻辑

现在再看紧凑版，就不神秘了：

```python
ypred = [predict(x) for x in xs]
```

意思是：4 条输入各预测一次。

```python
losses = [(yout - ygt) ** 2 for ygt, yout in zip(ys, ypred)]
```

意思是：标准答案和预测值一一配对，每条都算平方误差。

```python
total = sum(losses) * (1.0 / len(losses))
```

意思是：把每条的平方误差加起来，再除以样本数量，得到平均 loss。

```python
return total, ypred
```

意思是：返回总 loss，同时也把预测列表返回，方便我们观察模型现在猜成什么样。



## 4. 单步训练拆开看

一轮训练只有四步：

```text
1. forward：算 loss
2. zero_grad：清空旧梯度
3. backward：算新梯度
4. update：p.data += -learning_rate * p.grad
```

为什么要 `zero_grad`？因为 micrograd 里的梯度会用 `+=` 累加，不清空就会混入上一轮。

In [6]:
learning_rate = 0.05

current_loss, _ = loss()
model.zero_grad()
current_loss.backward()

print('before update loss:', current_loss.data)
print('first param before:', model.parameters()[0])

for p in model.parameters():
    p.data += -learning_rate * p.grad

new_loss, _ = loss()
print('first param after :', model.parameters()[0])
print('after update loss :', new_loss.data)

before update loss: 1.2506238100655815
first param before: Value(data=0.23550571390294128, grad=1.3017976446023733)
first param after : Value(data=0.1704158316728226, grad=1.3017976446023733)
after update loss : 1.0359526521059315


## 5. 完整训练循环

下面跑 40 步。loss 不一定每一步都严格下降，但整体应该往下。

In [7]:
random.seed(1337)
model = MLP(2, [4, 1])
learning_rate = 0.05
history = []

for step in range(40):
    total_loss, ypred = loss()
    history.append(total_loss.data)

    model.zero_grad()
    total_loss.backward()

    for p in model.parameters():
        p.data += -learning_rate * p.grad

    if step % 5 == 0 or step == 39:
        print(f'step {step:02d} loss={total_loss.data:.6f} predictions={[round(p.data, 3) for p in ypred]}')

final_loss, final_preds = loss()
print()
print('initial loss:', history[0])
print('final loss  :', final_loss.data)
assert final_loss.data < history[0]
print('OK: final loss is lower than initial loss.')

step 00 loss=1.250624 predictions=[0.414, 0.54, 0.177, 0.05]
step 05 loss=0.763476 predictions=[0.695, -0.115, 0.094, 0.01]
step 10 loss=0.615332 predictions=[0.768, -0.208, -0.032, 0.082]
step 15 loss=0.518375 predictions=[0.847, -0.292, -0.123, 0.117]
step 20 loss=0.477805 predictions=[1.024, -0.365, -0.148, 0.116]
step 25 loss=0.447024 predictions=[1.023, -0.432, -0.21, 0.083]
step 30 loss=0.423869 predictions=[1.073, -0.485, -0.232, 0.086]
step 35 loss=0.401922 predictions=[1.008, -0.531, -0.279, 0.069]
step 39 loss=0.388425 predictions=[1.008, -0.553, -0.293, 0.076]

initial loss: 1.2506238100655815
final loss  : 0.38775339679702936
OK: final loss is lower than initial loss.


## 6. 改一行实验

你现在可以只改一个东西：

```text
learning_rate = 0.01
learning_rate = 0.1
learning_rate = 1.0
```

观察：

```text
太小：下降很慢
合适：比较稳定下降
太大：可能震荡甚至变差
```

这就是学习率的直觉。

## 7. TODO Lab - 写一个参数更新函数

TODO：把参数更新这一步封装成函数。

你只要记住：

```text
新参数 = 旧参数 - learning_rate * grad
```

在 micrograd 代码里写成：

```python
p.data += -learning_rate * p.grad
```

In [8]:
def update_parameters(parameters, learning_rate):
    # TODO: 遍历 parameters，用 -learning_rate * grad 更新每个 p.data
    pass


qa_check_update_parameters(update_parameters)

TODO: 参数还没有变化，请先实现 update_parameters。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

一个参数的更新公式是 `p.data += -learning_rate * p.grad`。这里要对所有参数做同样的事。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def update_parameters(parameters, learning_rate):
    for p in parameters:
        p.data += -learning_rate * p.grad
```

</details>

## What To Remember

训练循环只记住六句话：

```text
1. model(x) 做前向预测。
2. loss 衡量预测和目标差多少。
3. zero_grad 清空上一轮梯度。
4. loss.backward() 计算每个参数的 grad。
5. 参数更新用 p.data += -learning_rate * p.grad。
6. 反复做很多轮，目标是让 loss 下降。
```

你前面学的所有导数、链式法则、Value、backward，最后都服务于第 4 步。

<!-- xiao-post:start -->
## 课后作业提交清单

这一节学完后，用下面 5 条自检：

```text
1. 我能按顺序讲出 forward、loss、zero_grad、backward、update。
2. 我能解释为什么每轮都要 zero_grad。
3. 我能手算一次 p.data += -lr * p.grad。
4. 我能解释 learning_rate 控制“每步走多大”。
5. 我跑过训练循环，并看到 loss 下降。
```


## AI 复盘检查 Prompt

```text
你是我的 micrograd 训练循环检查官。
我刚学完一个最小训练循环：forward、loss、zero_grad、backward、parameter update。
请你一次只问一个问题，检查我是否理解：
1. loss 是什么
2. grad 为什么能指导参数更新
3. 为什么更新方向是 -grad
4. learning_rate 是什么
5. 为什么每轮要 zero_grad
6. 如果 learning_rate 太大/太小会怎样
不要直接给答案。每次我回答后再评价，并给一个小数字例子。
```